# Modern Adult Income Benchmark

Exploratory look at the UCI Adult dataset, then a side-by-side comparison of logistic regression, random forests, AdaBoost (custom + sklearn), and histogram gradient boosting via the `income_classifier` package.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from income_classifier import load_adult, train_and_evaluate
from income_classifier.data import TARGET_COLUMN

sns.set_theme(style="whitegrid", context="notebook")


## Load & inspect

In [ ]:
df = load_adult()
print(df.shape)
df.head()


In [ ]:
df[TARGET_COLUMN].value_counts(normalize=True).round(3)

## Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
sns.histplot(data=df, x="age", hue=TARGET_COLUMN, bins=30, ax=axes[0], element="step")
sns.boxplot(data=df, x=TARGET_COLUMN, y="hours-per-week", ax=axes[1])
sns.boxplot(data=df, x=TARGET_COLUMN, y="education-num", ax=axes[2])
fig.tight_layout()


In [ ]:
cap = df.assign(has_capital_gain=df["capital-gain"] > 0)
pd.crosstab(cap["has_capital_gain"], cap[TARGET_COLUMN], normalize="index").round(3)


## Train the model zoo

Fits every estimator in the package and prints a leaderboard sorted by ROC-AUC.


In [ ]:
metrics, models, meta = train_and_evaluate(split="official", verbose=True)
print(meta["n_train"], "train /", meta["n_test"], "test")
metrics


In [ ]:
plot_df = metrics.set_index("model")[["roc_auc", "f1", "accuracy"]]
ax = plot_df.plot(kind="barh", figsize=(8, 4))
ax.set_xlabel("score")
ax.set_title("Adult income model comparison")
ax.invert_yaxis()
plt.tight_layout()
